# 🎙️ Batch Audio Transcription using Gemini 2.5 Pro

Reads all audio files from the `wav` folder, auto-detects language from subfolder names, and outputs clean paragraph transcripts.

## 1. Install Dependencies

In [ ]:
!pip install google-genai python-dotenv -q

## 2. Load API Key & Initialize Client

In [1]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

if not api_key or api_key == "your_gemini_api_key_here":
    raise ValueError("❌ Please set your GEMINI_API_KEY in the .env file!")

client = genai.Client(api_key=api_key)
print("✅ Gemini client initialized successfully!")

✅ Gemini client initialized successfully!


## 3. Configuration

In [2]:
# ============================================================
# 📁 WAV FOLDER PATH — Contains language subfolders
#    e.g., wav/hausa/, wav/igbo/, wav/pidgin/, wav/yoruba/
# ============================================================
WAV_FOLDER = r"D:\audio_transscript\wav"

# ============================================================
# ⚙️ MODEL
# ============================================================
MODEL_NAME = "gemini-3.1-pro-preview"

# Supported audio extensions
SUPPORTED_FORMATS = [".mp3", ".wav", ".aac", ".ogg", ".flac", ".m4a", ".wma", ".opus", ".webm"]

print(f"📁 WAV Folder : {WAV_FOLDER}")
print(f"🤖 Model      : {MODEL_NAME}")

📁 WAV Folder : D:\audio_transscript\wav
🤖 Model      : gemini-3.1-pro-preview


## 4. Scan All Audio Files

In [3]:
import os

# Scan wav folder — each subfolder name = language
audio_files = []

for lang_folder in sorted(os.listdir(WAV_FOLDER)):
    lang_path = os.path.join(WAV_FOLDER, lang_folder)
    if not os.path.isdir(lang_path):
        continue
    
    language = lang_folder.strip().capitalize()  # folder name = language
    
    for filename in sorted(os.listdir(lang_path)):
        ext = os.path.splitext(filename)[1].lower()
        if ext in SUPPORTED_FORMATS:
            filepath = os.path.join(lang_path, filename)
            size_mb = os.path.getsize(filepath) / (1024 * 1024)
            audio_files.append({
                "path": filepath,
                "filename": filename,
                "language": language,
                "size_mb": size_mb
            })

print(f"🔍 Found {len(audio_files)} audio files:\n")
for i, f in enumerate(audio_files, 1):
    print(f"  {i}. [{f['language']}] {f['filename']} ({f['size_mb']:.1f} MB)")

🔍 Found 12 audio files:

  1. [Hausa] Hausa Language_Sample 1 Audio.wav (35.5 MB)
  2. [Hausa] Hausa Language_Sample 2 Audio.wav (39.4 MB)
  3. [Hausa] Hausa Language_Sample 3 Audio.wav (50.8 MB)
  4. [Igbo] Igbo Language Audio_Sample 1_.wav (46.2 MB)
  5. [Igbo] Igbo Language Audio_Sample 2.wav (41.7 MB)
  6. [Igbo] Igbo Language Audio_Sample 3.wav (51.2 MB)
  7. [Pidgin] Pidgin English Audio_Sample 1.wav (40.7 MB)
  8. [Pidgin] Pidgin English_Audio 2.wav (46.0 MB)
  9. [Pidgin] Pidgin English_Audio 3.wav (54.6 MB)
  10. [Yoruba] Yoruba Language Audio_Sample 1.wav (60.2 MB)
  11. [Yoruba] Yoruba Language Audio_Sample 2.wav (43.6 MB)
  12. [Yoruba] Yoruba Language Audio_Sample 3-mc.wav (38.4 MB)


## 5. Transcribe All Audio Files

In [4]:
import time

results = []

for i, audio in enumerate(audio_files, 1):
    print(f"\n{'='*60}")
    print(f"[{i}/{len(audio_files)}] {audio['filename']}")
    print(f"  Language: {audio['language']} | Size: {audio['size_mb']:.1f} MB")
    print(f"{'='*60}")
    
    try:
        # Upload
        print("  ⬆️  Uploading...")
        uploaded_file = client.files.upload(file=audio["path"])
        
        # Build prompt — plain paragraph only, no timestamps, no speakers
        prompt = (
            f"Transcribe this audio file accurately. "
            f"The audio is in {audio['language']}. "
            f"Provide ONLY the plain text transcription in {audio['language']}. "
            f"Write it as a continuous paragraph with proper punctuation. "
            f"Do NOT include any timestamps. "
            f"Do NOT include any speaker labels or speaker identification. "
            f"Do NOT include any headers, titles, or formatting. "
            f"Just output the raw transcribed text as a paragraph."
        )
        
        # Transcribe
        print("  🧠 Transcribing...")
        start_time = time.time()
        
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=[uploaded_file, prompt]
        )
        
        elapsed = time.time() - start_time
        transcript = response.text.strip()
        
        results.append({
            "filename": audio["filename"],
            "language": audio["language"],
            "transcript": transcript
        })
        
        print(f"  ✅ Done in {elapsed:.1f}s")
        
        # Cleanup uploaded file
        try:
            client.files.delete(name=uploaded_file.name)
        except:
            pass
            
    except Exception as e:
        print(f"  ❌ Error: {e}")
        results.append({
            "filename": audio["filename"],
            "language": audio["language"],
            "transcript": f"ERROR: {str(e)}"
        })

print(f"\n\n🎉 Batch complete! {len(results)} files processed.")


[1/12] Hausa Language_Sample 1 Audio.wav
  Language: Hausa | Size: 35.5 MB
  ⬆️  Uploading...
  🧠 Transcribing...
  ✅ Done in 59.3s

[2/12] Hausa Language_Sample 2 Audio.wav
  Language: Hausa | Size: 39.4 MB
  ⬆️  Uploading...
  🧠 Transcribing...
  ✅ Done in 109.3s

[3/12] Hausa Language_Sample 3 Audio.wav
  Language: Hausa | Size: 50.8 MB
  ⬆️  Uploading...
  🧠 Transcribing...
  ✅ Done in 56.3s

[4/12] Igbo Language Audio_Sample 1_.wav
  Language: Igbo | Size: 46.2 MB
  ⬆️  Uploading...
  🧠 Transcribing...
  ✅ Done in 149.1s

[5/12] Igbo Language Audio_Sample 2.wav
  Language: Igbo | Size: 41.7 MB
  ⬆️  Uploading...
  🧠 Transcribing...
  ✅ Done in 54.4s

[6/12] Igbo Language Audio_Sample 3.wav
  Language: Igbo | Size: 51.2 MB
  ⬆️  Uploading...
  🧠 Transcribing...
  ✅ Done in 147.9s

[7/12] Pidgin English Audio_Sample 1.wav
  Language: Pidgin | Size: 40.7 MB
  ⬆️  Uploading...
  🧠 Transcribing...
  ✅ Done in 75.4s

[8/12] Pidgin English_Audio 2.wav
  Language: Pidgin | Size: 46.0 MB


## 6. View All Transcriptions

In [13]:
for r in results:
    print(f"\n{'='*70}")
    print(f"📁 {r['filename']}  |  🌐 {r['language']}")
    print(f"{'='*70}")
    print(r["transcript"])
    print()


📁 Hausa Language_Sample 1 Audio.wav  |  🌐 Hausa
Dama asalin zakka akan zinari da azurfa aka fara faralanta ta, kafin a dawo kan naira da dala da sauran su su kimantawa ne ake yi. To malamai inda suka samu sabani shine, idan zinarin da azurfan ba kasuwanci ake yi da shi ba, ba an ajiye shi ne a matsayin jari ba, a'a, an ajiye shi ne don mace kawai tayi ado da shi tayi kwalliya, shi ma akwai zakka a kan sa ko babu zakka a kan shi? To farko dai, a yanzu mata masu sayen gulagulai, in Allah ya yarda, kashi 95 cikin 100 nasu ba suna sayen gwal ne saboda suyi kwalliya da shi kawai ba, suna sayen gwal ne a matsayin jari, a matsayin hanyar aje kudi kuma kudin ya karu. Saboda haka mafi yawa yanzu ana aje sarakoki da mundaye na gwal, yanzu har ba'a yi katan gwamnati. Yanzu duka an rufa akan me? Akan gwal. Saboda haka duka irin wa'innan akwai zakka akan su, su babu ma wani kokwanto ko dawo da hankali malamai a kai. Inda ake da sabani a wajen malamai shine, wacce ita amfanin gwal din a wajenta kaw

## 7. Save Transcriptions to CSV

In [6]:
import csv

output_csv = os.path.join(WAV_FOLDER, "..", "transcriptions_gemini_3.1_pro.csv")
output_csv = os.path.abspath(output_csv)

with open(output_csv, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["filename", "language", "transcript"])
    writer.writeheader()
    writer.writerows(results)

print(f"💾 Saved {len(results)} transcriptions to:")
print(f"   {output_csv}")

💾 Saved 12 transcriptions to:
   D:\audio_transscript\transcriptions_gemini_3.1_pro.csv


In [8]:
# ============================================================
# Save Gemini 3.1 Pro transcriptions into the Excel file
# as a new column after "Gemini -2.5- Pro"
# ============================================================
import pandas as pd

EXCEL_FILE = r"D:\audio_transscript\gemnai\MCF Languages.xlsx"
NEW_COL = "Gemini -3.1- Pro"

# Map language folder names to sheet names
LANG_TO_SHEET = {
    "Hausa": "Hausa",
    "Igbo": "Igbo",
    "Yoruba": "Yoruba",
    "Pidgin": "Pidgin ",   # note trailing space in sheet name
}

# Group results by language
from collections import defaultdict
lang_transcripts = defaultdict(list)
for r in results:
    lang_transcripts[r["language"]].append(r["transcript"])

# Read all sheets, add the new column, write back
all_sheets = pd.read_excel(EXCEL_FILE, sheet_name=None)

for lang, sheet_name in LANG_TO_SHEET.items():
    if sheet_name not in all_sheets:
        print(f"  SKIP: sheet '{sheet_name}' not found")
        continue

    df = all_sheets[sheet_name]
    transcripts = lang_transcripts.get(lang, [])

    # Pad with NaN if fewer transcripts than rows
    padded = transcripts + [None] * (len(df) - len(transcripts))
    padded = padded[:len(df)]  # trim if more

    # Insert after "Gemini -2.5- Pro" column
    if "Gemini -2.5- Pro" in df.columns:
        insert_pos = df.columns.get_loc("Gemini -2.5- Pro") + 1
    else:
        insert_pos = len(df.columns)

    # Remove if already exists (re-run safe)
    if NEW_COL in df.columns:
        df = df.drop(columns=[NEW_COL])

    df.insert(insert_pos, NEW_COL, padded)
    all_sheets[sheet_name] = df
    print(f"  OK: {sheet_name.strip()} - added '{NEW_COL}' ({len(transcripts)} transcripts)")

# Write all sheets back
with pd.ExcelWriter(EXCEL_FILE, engine="openpyxl") as writer:
    for sheet_name, df in all_sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"\nDONE! Updated: {EXCEL_FILE}")


  OK: Hausa - added 'Gemini -3.1- Pro' (3 transcripts)
  OK: Igbo - added 'Gemini -3.1- Pro' (3 transcripts)
  OK: Yoruba - added 'Gemini -3.1- Pro' (3 transcripts)
  OK: Pidgin - added 'Gemini -3.1- Pro' (3 transcripts)

DONE! Updated: D:\audio_transscript\gemnai\MCF Languages.xlsx
